## Reservando os dados certos (Reservoir sampling)

In [14]:
import string

ascii_uppercase_letters = list(string.ascii_uppercase)
ascii_lowercase_letters = list(string.ascii_lowercase)
data_stream = ascii_uppercase_letters + ascii_lowercase_letters
print(data_stream)

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [2]:
from random import seed, randint
seed(9) # change this value for different results
sample_size = 5
sample = []

In [3]:
for index, element in enumerate(data_stream):
        # Until the reservoir is filled, we add elements
        if index < sample_size:
                sample.append(element)
        else:
                # Having filled the reservoir, we test a
                # random replacement based on the elements
                # seen in the data stream
                drawn = randint(0, index)
                # If the drawn number is less or equal the
                # sample size, we replace a previous element
                # with the one arriving from the stream
                if drawn < sample_size:
                        sample[drawn] = element

In [4]:
print(sample)

['y', 'e', 'v', 'F', 'i']


## Filtrando elementos de fluxo pelo núcleo (Bloom Filter)

In [5]:
hash_functions = 3
bit_vector_length = 10
bit_vector = [0] * bit_vector_length

In [6]:
from hashlib import md5, sha1

def hash_fn(element, i, length):
    """ This is a magic function """
    h1 = int(md5(element.encode('ascii')).hexdigest(),16)
    h2 = int(sha1(element.encode('ascii')).hexdigest(),16)
    return (h1 + i * h2) % length

def insert_filter(website):
    result = list()
    for hash_number in range(hash_functions):
        position = hash_fn(website, hash_number, 
                          bit_vector_length)
        result.append(position)
        bit_vector[position] = 1
    print ('Inserted in positions: %s' % result)

def check_filter(website):
    result = list()
    for hash_number in range(hash_functions):
        position = hash_fn(website, hash_number, 
                          bit_vector_length)
        result.append((position,bit_vector[position]))
    print ('Bytes in positions: %s' % result)

In [7]:
insert_filter('wikipedia.org')
print(bit_vector)

Inserted in positions: [0, 8, 6]
[1, 0, 0, 0, 0, 0, 1, 0, 1, 0]


In [10]:
insert_filter('youtube.com')
print(bit_vector)

Inserted in positions: [3, 0, 7]
[1, 0, 0, 1, 0, 0, 1, 1, 1, 0]


In [20]:
check_filter('yahoo.com')

Bytes in positions: [(7, 1), (5, 0), (3, 1)]


O número $k$ ideal de funções hash para usar para minimizar colisões pode então ser estimado usando a fórmula a seguir (ln é o logaritmo natural):

$k = \frac{m}{n} \cdot \ln(2)$

Depois de definir $m$, $n$ e $k$, esta segunda fórmula ajuda a estimar a probabilidade de uma colisão (uma taxa de falso positivo) usando um filtro de Bloom:

$\text{taxa de falso positivo} = (1 - exp(\frac{-kn}{m}))^k$

Sendo

- $n$ = o número de objetos distintos que é possível esperar adicionar
- $m$ = tamanho do vetor de bits (que é igual ao espaço de memória)
- $k$ = o número de funções hash (que é igual ao tempo)


HyperLogLog

1. Um hash converte cada elemento recebido do fluxo em um número.

2. O algoritmo converte o número em binário, o padrão numérico base 2 que os computadores usam.

3. O algoritmo conta o número de zeros iniciais no número binário e moni-tora o número máximo que vê, que é $n$.

4. O algoritmo estima o número de elementos distintos passados no fluxo usando $n$. O número de elementos distintos é $2^n$.

Count-Min Sketch

1. Inicialize todos os vetores de bits com zero em todas as posições.

2. Aplique a função hash para cada vetor de bits quando receber um objeto de um fluxo. Use o endereço numérico resultante para incrementar o valor nesta posição.

3. Aplique a função hash a um objeto e recupere o valor na posição associada quando for pedido para estimar a frequência de um objeto. De todos os valores recebidos dos vetores de bits, pegue o menor conforme a frequência do fluxo.